### Task 1: Build a Complete ETL Pipeline

Build an ETL pipeline that processes raw e-commerce order data into a clean analytical dataset.

**Step 1 — Generate raw data:** Create a list of 200 raw order records as dictionaries with these fields: `order_id`, `customer_id`, `product_name`, `quantity`, `unit_price`, `order_date`, `shipping_country`. Intentionally include at least 15 problematic records across these categories:
- 3 records with missing `product_name` (None or empty string)
- 3 records with negative `quantity` or `unit_price`
- 3 records with malformed `order_date` (e.g., "not-a-date", "2025-13-45")
- 3 records with duplicate `order_id` values
- 3 records where `quantity` or `unit_price` is a string instead of a number

In [1]:
import random
from datetime import datetime, timedelta
random.seed(123)
products = ["Laptop", "Phone", "Tablet", "Keyboard", "Mouse", "Monitor"]
countries = ["usa", "germany", "france", "canada", "japan"]

raw_records = []

for i in range(1, 201):
    record = {
        "order_id": i,
        "customer_id": random.randint(1, 50),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 500), 2),
        "order_date": (datetime(2025,1,1) + timedelta(days=random.randint(0,120))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    }
    raw_records.append(record)


raw_records[5]["product_name"] = None
raw_records[15]["product_name"] = ""
raw_records[25]["product_name"] = None


raw_records[35]["quantity"] = -2
raw_records[45]["unit_price"] = -50
raw_records[55]["quantity"] = -1

raw_records[65]["order_date"] = "not-a-date"
raw_records[75]["order_date"] = "2025-13-45"
raw_records[85]["order_date"] = "wrong-date"


raw_records[95]["order_id"] = 10
raw_records[105]["order_id"] = 20
raw_records[115]["order_id"] = 30


raw_records[125]["quantity"] = "five"
raw_records[135]["unit_price"] = "ten"
raw_records[145]["quantity"] = "seven"

print(len(raw_records))
print(raw_records[:5])

200
[{'order_id': 1, 'customer_id': 4, 'product_name': 'Tablet', 'quantity': 1, 'unit_price': 386.79, 'order_date': '2025-02-04', 'shipping_country': 'usa'}, {'order_id': 2, 'customer_id': 3, 'product_name': 'Keyboard', 'quantity': 5, 'unit_price': 285.54, 'order_date': '2025-02-13', 'shipping_country': 'usa'}, {'order_id': 3, 'customer_id': 11, 'product_name': 'Phone', 'quantity': 3, 'unit_price': 284.87, 'order_date': '2025-03-31', 'shipping_country': 'germany'}, {'order_id': 4, 'customer_id': 11, 'product_name': 'Laptop', 'quantity': 4, 'unit_price': 389.06, 'order_date': '2025-04-23', 'shipping_country': 'japan'}, {'order_id': 5, 'customer_id': 25, 'product_name': 'Laptop', 'quantity': 1, 'unit_price': 164.57, 'order_date': '2025-02-27', 'shipping_country': 'usa'}]


#### In this step, I simulated a real-world scenario where data isn't perfect. I generated 200 order records but intentionally "poisoned" the well by injecting 15+ broken entries. This includes missing product names, negative prices, and dates that don't exist (like March 45th).

**Step 2 — Extract with validation:** Write an `extract(raw_records)` function that:
- Parses each record and validates required fields exist
- Validates that `quantity` and `unit_price` are positive numbers
- Validates that `order_date` is a parseable date
- Returns two lists: `valid_records` and `rejected_records`
- Each rejected record should include the reason for rejection
- Print a summary: count of valid, count of rejected by reason

In [2]:
import pandas as pd
from collections import Counter

def extract(raw_records):

    valid_records = []
    rejected_records = []
    order_ids = set()

    for record in raw_records:

        try:

            required_fields = [
                "order_id","customer_id","product_name",
                "quantity","unit_price","order_date","shipping_country"
            ]

            if not all(field in record for field in required_fields):
                rejected_records.append({"record": record, "reason": "missing field"})
                continue

            if record["product_name"] is None or record["product_name"] == "":
                rejected_records.append({"record": record, "reason": "missing product_name"})
                continue

            quantity = float(record["quantity"])
            price = float(record["unit_price"])

            if quantity <= 0 or price <= 0:
                rejected_records.append({"record": record, "reason": "negative quantity or price"})
                continue

            try:
                pd.to_datetime(record["order_date"])
            except:
                rejected_records.append({"record": record, "reason": "invalid order_date"})
                continue

            if record["order_id"] in order_ids:
                rejected_records.append({"record": record, "reason": "duplicate order_id"})
                continue

            order_ids.add(record["order_id"])
            valid_records.append(record)

        except:
            rejected_records.append({"record": record, "reason": "invalid data type"})

    reason_counts = Counter(r["reason"] for r in rejected_records)

    print("Valid records:", len(valid_records))
    print("Rejected records:", len(rejected_records))

    for reason, count in reason_counts.items():
        print(reason, ":", count)

    return valid_records, rejected_records
valid, rejected = extract(raw_records)

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
duplicate order_id : 3
invalid data type : 3


**Step 3 — Transform:** Write a `transform(valid_records)` function that:
- Converts to a pandas DataFrame
- Computes `total_amount = quantity * unit_price`
- Extracts `order_month` and `order_day_of_week` from the date
- Standardizes `shipping_country` to title case
- Removes duplicate `order_id` values (keep the first)
- Adds an `amount_category` column: "small" (<$25), "medium" ($25-$100), "large" (>$100)
- Returns the cleaned DataFrame

In [3]:
import pandas as pd

def transform(valid_records):
   
    df = pd.DataFrame(valid_records)
    
    
    df['total_amount'] = df['quantity'] * df['unit_price']
    
    
    df['order_date'] = pd.to_datetime(df['order_date'], format='%Y-%m-%d', errors='coerce')
    
    
    df['order_month'] = df['order_date'].dt.month
    df['order_day_of_week'] = df['order_date'].dt.day_name()
    
    
    df['shipping_country'] = df['shipping_country'].str.title()
    
    
    df = df.drop_duplicates(subset='order_id', keep='first')
    
   
    def categorize_amount(amount):
        if amount < 25:
            return "small"
        elif amount <= 100:
            return "medium"
        else:
            return "large"
    
    df['amount_category'] = df['total_amount'].apply(categorize_amount)
    
    
    return df


cleaned_df = transform(valid)
print(cleaned_df.head())

   order_id  customer_id product_name  quantity  unit_price order_date  \
0         1            4       Tablet         1      386.79 2025-02-04   
1         2            3     Keyboard         5      285.54 2025-02-13   
2         3           11        Phone         3      284.87 2025-03-31   
3         4           11       Laptop         4      389.06 2025-04-23   
4         5           25       Laptop         1      164.57 2025-02-27   

  shipping_country  total_amount  order_month order_day_of_week  \
0              Usa        386.79            2           Tuesday   
1              Usa       1427.70            2          Thursday   
2          Germany        854.61            3            Monday   
3            Japan       1556.24            4         Wednesday   
4              Usa        164.57            2          Thursday   

  amount_category  
0           large  
1           large  
2           large  
3           large  
4           large  


**Step 4 — Load:** Write a `load(df, path)` function that saves the DataFrame to a Parquet file. Then read it back and verify the row count and dtypes match.

In [4]:
import pandas as pd

def load(df, path):
    """
    Saves the DataFrame to a Parquet file, then reads it back
    and verifies row count and dtypes.
    """
   
    df.to_parquet(path, engine='pyarrow', index=False)
    print(f"DataFrame saved to {path}")
    
   
    df_loaded = pd.read_parquet(path, engine='pyarrow')
    print(f"DataFrame loaded from {path}")
    
   
    if len(df) == len(df_loaded):
        print(f"✅ Row count matches: {len(df)}")
    else:
        print(f"⚠ Row count mismatch! Original: {len(df)}, Loaded: {len(df_loaded)}")
    
    
    print("\nOriginal dtypes:")
    print(df.dtypes)
    print("\nLoaded dtypes:")
    print(df_loaded.dtypes)
    
    return df_loaded


cleaned_df = transform(valid)
loaded_df = load(cleaned_df, "cleaned_orders.parquet")

DataFrame saved to cleaned_orders.parquet
DataFrame loaded from cleaned_orders.parquet
✅ Row count matches: 185

Original dtypes:
order_id                      int64
customer_id                   int64
product_name                    str
quantity                      int64
unit_price                  float64
order_date           datetime64[us]
shipping_country                str
total_amount                float64
order_month                   int32
order_day_of_week               str
amount_category                 str
dtype: object

Loaded dtypes:
order_id                      int64
customer_id                   int64
product_name                    str
quantity                      int64
unit_price                  float64
order_date           datetime64[us]
shipping_country                str
total_amount                float64
order_month                   int32
order_day_of_week               str
amount_category                 str
dtype: object


**Step 5:** Run the full pipeline: `extract → transform → load`. Print a summary showing records at each stage (raw → valid → transformed → loaded).

In [5]:

valid_records, rejected_records = extract(raw_records)

print("\n--- Extraction Summary ---")
print(f"Raw records: {len(raw_records)}")
print(f"Valid records: {len(valid_records)}")
print(f"Rejected records: {len(rejected_records)}")


cleaned_df = transform(valid_records)

print("\n--- Transform Summary ---")
print(f"Records after transform (duplicates removed): {len(cleaned_df)}")
print("Columns:", cleaned_df.columns.tolist())


loaded_df = load(cleaned_df, "cleaned_orders.parquet")

print("\n--- Load Summary ---")
print(f"Loaded records: {len(loaded_df)}")
print("Loaded columns:", loaded_df.columns.tolist())


print("\nFirst 5 rows of loaded DataFrame:")
print(loaded_df.head())

Valid records: 185
Rejected records: 15
missing product_name : 3
negative quantity or price : 3
invalid order_date : 3
duplicate order_id : 3
invalid data type : 3

--- Extraction Summary ---
Raw records: 200
Valid records: 185
Rejected records: 15

--- Transform Summary ---
Records after transform (duplicates removed): 185
Columns: ['order_id', 'customer_id', 'product_name', 'quantity', 'unit_price', 'order_date', 'shipping_country', 'total_amount', 'order_month', 'order_day_of_week', 'amount_category']
DataFrame saved to cleaned_orders.parquet
DataFrame loaded from cleaned_orders.parquet
✅ Row count matches: 185

Original dtypes:
order_id                      int64
customer_id                   int64
product_name                    str
quantity                      int64
unit_price                  float64
order_date           datetime64[us]
shipping_country                str
total_amount                float64
order_month                   int32
order_day_of_week               str


### Task 2: ETL vs ELT Comparison

**Step 1:** Implement an ELT variant of the pipeline from Task 1:
- Extract: do minimal validation (only reject unparseable JSON/records)
- Load: save the raw (mostly unvalidated) data to a Parquet "data lake" file
- Transform: read from the data lake file, apply the same validation and transformation rules as Task 1

In [6]:

import pandas as pd
from datetime import datetime, timedelta
import random
from collections import Counter


random.seed(123)
products = ["Laptop", "Phone", "Tablet", "Keyboard", "Mouse", "Monitor"]
countries = ["usa", "germany", "france", "canada", "japan"]

raw_records = []
for i in range(1, 201):
    record = {
        "order_id": i,
        "customer_id": random.randint(1, 50),
        "product_name": random.choice(products),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 500), 2),
        "order_date": (datetime(2025,1,1) + timedelta(days=random.randint(0,120))).strftime("%Y-%m-%d"),
        "shipping_country": random.choice(countries)
    }
    raw_records.append(record)


raw_records[5]["product_name"] = None
raw_records[15]["product_name"] = ""
raw_records[25]["product_name"] = None

raw_records[35]["quantity"] = -2
raw_records[45]["unit_price"] = -50
raw_records[55]["quantity"] = -1

raw_records[65]["order_date"] = "not-a-date"
raw_records[75]["order_date"] = "2025-13-45"
raw_records[85]["order_date"] = "wrong-date"

raw_records[95]["order_id"] = 10
raw_records[105]["order_id"] = 20
raw_records[115]["order_id"] = 30

raw_records[125]["quantity"] = "five"
raw_records[135]["unit_price"] = "ten"
raw_records[145]["quantity"] = "seven"


def elt_extract(raw_records):
    valid_json_records = []
    rejected_records = []
    
    for record in raw_records:
        if isinstance(record, dict):
            valid_json_records.append(record)
        else:
            rejected_records.append({"record": record, "reason": "invalid JSON / not a dict"})
    
    print(f"ELT Extract: valid JSON records = {len(valid_json_records)}, rejected = {len(rejected_records)}")
    return valid_json_records, rejected_records


def elt_load(raw_records, path="raw_data_lake.parquet"):
    """
    Save raw (mostly unvalidated) data to Parquet (data lake) without type conversion.
    """
    df_raw = pd.DataFrame(raw_records)
    
    
    for col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(str)
    
    df_raw.to_parquet(path, engine="pyarrow", index=False)
    print(f"ELT Load: saved raw data to {path}")
    return path


def elt_transform(path="raw_data_lake.parquet"):
    df = pd.read_parquet(path, engine="pyarrow")
    
    valid_records = []
    rejected_records = []
    order_ids = set()
    
    for _, record in df.iterrows():
        error_reasons = []
        
        
        required_fields = ["order_id","customer_id","product_name","quantity",
                           "unit_price","order_date","shipping_country"]
        for field in required_fields:
            if field not in record or pd.isna(record[field]) or str(record[field]).strip() == "":
                error_reasons.append(f"Missing or empty field: {field}")
        
       
        try:
            qty = float(record['quantity'])
            price = float(record['unit_price'])
            if qty <= 0: error_reasons.append("Quantity must be positive")
            if price <= 0: error_reasons.append("Unit price must be positive")
        except:
            error_reasons.append("Quantity or unit_price is not a valid number")
        
       
        try:
            pd.to_datetime(record['order_date'], format="%Y-%m-%d")
        except:
            error_reasons.append(f"Malformed order_date: {record['order_date']}")
        
        
        if record["order_id"] in order_ids:
            error_reasons.append("Duplicate order_id")
        else:
            order_ids.add(record["order_id"])
        
        if error_reasons:
            rejected_records.append({"record": record.to_dict(), "reasons": error_reasons})
        else:
            valid_records.append(record.to_dict())
    
    print(f"ELT Transform: valid records = {len(valid_records)}, rejected records = {len(rejected_records)}")
    
    
    cleaned_df = pd.DataFrame(valid_records)
    if not cleaned_df.empty:
        cleaned_df['quantity'] = cleaned_df['quantity'].astype(float)
        cleaned_df['unit_price'] = cleaned_df['unit_price'].astype(float)
        cleaned_df['total_amount'] = cleaned_df['quantity'] * cleaned_df['unit_price']
        cleaned_df['order_date'] = pd.to_datetime(cleaned_df['order_date'], format='%Y-%m-%d', errors='coerce')
        cleaned_df['order_month'] = cleaned_df['order_date'].dt.month
        cleaned_df['order_day_of_week'] = cleaned_df['order_date'].dt.day_name()
        cleaned_df['shipping_country'] = cleaned_df['shipping_country'].str.title()
        cleaned_df = cleaned_df.drop_duplicates(subset='order_id', keep='first')
        
        def categorize_amount(amount):
            if amount < 25:
                return "small"
            elif amount <= 100:
                return "medium"
            else:
                return "large"
        cleaned_df['amount_category'] = cleaned_df['total_amount'].apply(categorize_amount)
    
    return cleaned_df, rejected_records

valid_json, rejected_json = elt_extract(raw_records)


parquet_path = elt_load(valid_json, path="raw_data_lake.parquet")


cleaned_df_elt, rejected_elt = elt_transform(parquet_path)

print("\nELT Pipeline Summary:")
print(f"Raw JSON records: {len(valid_json)}")
print(f"Rejected during extract: {len(rejected_json)}")
print(f"Rejected during transform: {len(rejected_elt)}")
print(f"Final cleaned records: {len(cleaned_df_elt)}")


print("\nFirst 5 rows of ELT cleaned DataFrame:")
print(cleaned_df_elt.head())

ELT Extract: valid JSON records = 200, rejected = 0
ELT Load: saved raw data to raw_data_lake.parquet
ELT Transform: valid records = 185, rejected records = 15

ELT Pipeline Summary:
Raw JSON records: 200
Rejected during extract: 0
Rejected during transform: 15
Final cleaned records: 185

First 5 rows of ELT cleaned DataFrame:
  order_id customer_id product_name  quantity  unit_price order_date  \
0        1           4       Tablet       1.0      386.79 2025-02-04   
1        2           3     Keyboard       5.0      285.54 2025-02-13   
2        3          11        Phone       3.0      284.87 2025-03-31   
3        4          11       Laptop       4.0      389.06 2025-04-23   
4        5          25       Laptop       1.0      164.57 2025-02-27   

  shipping_country  total_amount  order_month order_day_of_week  \
0              Usa        386.79            2           Tuesday   
1              Usa       1427.70            2          Thursday   
2          Germany        854.61     

**Step 2:** Compare ETL and ELT approaches. Write a markdown cell addressing:
- How many records made it to the destination in each approach?
- At what stage were problems caught in each?
- What are the advantages of each approach?
- When would you choose one over the other?

### ETL vs ELT Comparison

**Records in destination:**  
- **ETL:** Only fully validated and cleaned records reach the destination.  
- **ELT:** Raw data is first loaded; validation happens later, so all data initially lands in the data lake.  

**Problem detection stage:**  
- **ETL:** Issues are caught during extraction/validation before loading.  
- **ELT:** Issues are caught during transformation after loading.  

**Advantages:**  
- **ETL:** Cleaner data in destination, fewer downstream errors.  
- **ELT:** Faster initial loading, easy to reprocess or apply new rules on raw data.  

**When to choose:**  
- **ETL:** When data quality is critical and destination systems need clean data.  
- **ELT:** When working with big data lakes, want flexibility to transform later, or need to keep all raw data.

### Task 3: Simulate Modes of Dataflow

Model a simplified analytics system with three components: an order ingestion service, a feature computation service, and a prediction service.

**Step 1 — Data passing through a database:** Simulate a shared database using a dictionary. The order service writes new orders to the database. The feature service reads orders and computes features (total orders, average amount, last order date per customer). The prediction service reads the features.

```python
# Shared database (simulated)
database = {"orders": [], "features": {}}

def order_service_write(order):
    database["orders"].append(order)

def feature_service_compute():
    # Read from database, compute features, write back
    ...

def prediction_service_read(customer_id):
    # Read features from database
    ...
```

In [7]:

from datetime import datetime


database = {
    "orders": [],      
    "features": {}     
}


def order_service_write(order):
    """
    Simulate writing a new order to the database.
    order: dict with keys ['order_id','customer_id','quantity','unit_price','order_date']
    """
    database["orders"].append(order)
    print(f"Order added: {order}")


def feature_service_compute():
    """
    Read all orders and compute features per customer:
    - total_orders
    - average_amount
    - last_order_date
    Store features back into database['features']
    """
    features = {}
    for order in database["orders"]:
        cid = order["customer_id"]
        total_amount = order["quantity"] * order["unit_price"]
        order_date = datetime.strptime(order["order_date"], "%Y-%m-%d")
        
        if cid not in features:
            features[cid] = {
                "total_orders": 1,
                "total_amount_sum": total_amount,
                "last_order_date": order_date
            }
        else:
            features[cid]["total_orders"] += 1
            features[cid]["total_amount_sum"] += total_amount
            if order_date > features[cid]["last_order_date"]:
                features[cid]["last_order_date"] = order_date
    
    
    for cid, f in features.items():
        f["average_amount"] = f["total_amount_sum"] / f["total_orders"]
        del f["total_amount_sum"] 

    database["features"] = features
    print("Features computed and stored in database.")


def prediction_service_read(customer_id):
    """
    Read features for a customer from the database.
    Returns a dict of features or None if customer not found.
    """
    return database["features"].get(customer_id, None)



order_service_write({"order_id": 1, "customer_id": 101, "quantity": 2, "unit_price": 50, "order_date": "2025-03-01"})
order_service_write({"order_id": 2, "customer_id": 102, "quantity": 1, "unit_price": 100, "order_date": "2025-03-02"})
order_service_write({"order_id": 3, "customer_id": 101, "quantity": 1, "unit_price": 75, "order_date": "2025-03-05"})


feature_service_compute()


customer_features = prediction_service_read(101)
print("\nFeatures for customer 101:")
print(customer_features)

Order added: {'order_id': 1, 'customer_id': 101, 'quantity': 2, 'unit_price': 50, 'order_date': '2025-03-01'}
Order added: {'order_id': 2, 'customer_id': 102, 'quantity': 1, 'unit_price': 100, 'order_date': '2025-03-02'}
Order added: {'order_id': 3, 'customer_id': 101, 'quantity': 1, 'unit_price': 75, 'order_date': '2025-03-05'}
Features computed and stored in database.

Features for customer 101:
{'total_orders': 2, 'last_order_date': datetime.datetime(2025, 3, 5, 0, 0), 'average_amount': 87.5}


**Step 2 — Data passing through services:** Refactor so the prediction service requests data from the feature service directly (function call simulating an API request). The feature service requests raw orders from the order service.

In [8]:

from datetime import datetime


orders_storage = []

def order_service_write(order):
    """Simulate writing a new order to the order service."""
    orders_storage.append(order)
    print(f"Order added: {order}")

def order_service_get_all():
    """Simulate API call to get all orders."""
    return orders_storage

def feature_service_compute():
    """
    Simulate feature computation by requesting orders from the order service.
    Computes per-customer features:
    - total_orders
    - average_amount
    - last_order_date
    Returns a dict of features.
    """
    orders = order_service_get_all() 
    features = {}

    for order in orders:
        cid = order["customer_id"]
        total_amount = order["quantity"] * order["unit_price"]
        order_date = datetime.strptime(order["order_date"], "%Y-%m-%d")

        if cid not in features:
            features[cid] = {
                "total_orders": 1,
                "total_amount_sum": total_amount,
                "last_order_date": order_date
            }
        else:
            features[cid]["total_orders"] += 1
            features[cid]["total_amount_sum"] += total_amount
            if order_date > features[cid]["last_order_date"]:
                features[cid]["last_order_date"] = order_date

    for cid, f in features.items():
        f["average_amount"] = f["total_amount_sum"] / f["total_orders"]
        del f["total_amount_sum"]

    return features

def prediction_service_get(customer_id):
    """
    Request features from the feature service for a specific customer.
    Returns features or None.
    """
    features = feature_service_compute() 
    return features.get(customer_id, None)

order_service_write({"order_id": 1, "customer_id": 101, "quantity": 2, "unit_price": 50, "order_date": "2025-03-01"})
order_service_write({"order_id": 2, "customer_id": 102, "quantity": 1, "unit_price": 100, "order_date": "2025-03-02"})
order_service_write({"order_id": 3, "customer_id": 101, "quantity": 1, "unit_price": 75, "order_date": "2025-03-05"})

customer_features = prediction_service_get(101)
print("\nFeatures for customer 101:")
print(customer_features)

Order added: {'order_id': 1, 'customer_id': 101, 'quantity': 2, 'unit_price': 50, 'order_date': '2025-03-01'}
Order added: {'order_id': 2, 'customer_id': 102, 'quantity': 1, 'unit_price': 100, 'order_date': '2025-03-02'}
Order added: {'order_id': 3, 'customer_id': 101, 'quantity': 1, 'unit_price': 75, 'order_date': '2025-03-05'}

Features for customer 101:
{'total_orders': 2, 'last_order_date': datetime.datetime(2025, 3, 5, 0, 0), 'average_amount': 87.5}


**Step 3 — Data passing through a message broker:** Implement a simple pubsub broker (a class with `publish`, `subscribe`, and `get_latest` methods). The order service publishes new orders to an "orders" topic. The feature service subscribes and computes features whenever new orders arrive, publishing them to a "features" topic. The prediction service subscribes to "features."

In [9]:
from datetime import datetime


class Broker:
    def __init__(self):
        self.topics = {}  
        self.subscribers = {}  

    def publish(self, topic, message):
        """Publish a message to a topic."""
        if topic not in self.topics:
            self.topics[topic] = []
        self.topics[topic].append(message)
        # Notify subscribers
        for callback in self.subscribers.get(topic, []):
            callback(message)

    def subscribe(self, topic, callback):
        """Subscribe a callback to a topic."""
        if topic not in self.subscribers:
            self.subscribers[topic] = []
        self.subscribers[topic].append(callback)

    def get_latest(self, topic):
        """Return all messages from a topic."""
        return self.topics.get(topic, [])


broker = Broker()


features_storage = {}  

def feature_service_on_order(order):
    """Callback when a new order arrives."""
    cid = order["customer_id"]
    total_amount = order["quantity"] * order["unit_price"]
    order_date = datetime.strptime(order["order_date"], "%Y-%m-%d")

    if cid not in features_storage:
        features_storage[cid] = {
            "total_orders": 1,
            "total_amount_sum": total_amount,
            "last_order_date": order_date
        }
    else:
        f = features_storage[cid]
        f["total_orders"] += 1
        f["total_amount_sum"] += total_amount
        if order_date > f["last_order_date"]:
            f["last_order_date"] = order_date

    f = features_storage[cid]
    f["average_amount"] = f["total_amount_sum"] / f["total_orders"]

    broker.publish("features", {cid: features_storage[cid]})

broker.subscribe("orders", feature_service_on_order)


latest_features = {}

def prediction_service_on_features(feature_update):
    """Callback when features are updated."""
    latest_features.update(feature_update)

broker.subscribe("features", prediction_service_on_features)


def order_service_publish(order):
    """Publish a new order to the 'orders' topic."""
    broker.publish("orders", order)
    print(f"Order published: {order}")


order_service_publish({"order_id": 1, "customer_id": 101, "quantity": 2, "unit_price": 50, "order_date": "2025-03-01"})
order_service_publish({"order_id": 2, "customer_id": 102, "quantity": 1, "unit_price": 100, "order_date": "2025-03-02"})
order_service_publish({"order_id": 3, "customer_id": 101, "quantity": 1, "unit_price": 75, "order_date": "2025-03-05"})

print("\nLatest features known to prediction service:")
print(latest_features)

print("\nAll orders in broker 'orders' topic:")
print(broker.get_latest("orders"))

Order published: {'order_id': 1, 'customer_id': 101, 'quantity': 2, 'unit_price': 50, 'order_date': '2025-03-01'}
Order published: {'order_id': 2, 'customer_id': 102, 'quantity': 1, 'unit_price': 100, 'order_date': '2025-03-02'}
Order published: {'order_id': 3, 'customer_id': 101, 'quantity': 1, 'unit_price': 75, 'order_date': '2025-03-05'}

Latest features known to prediction service:
{101: {'total_orders': 2, 'total_amount_sum': 175, 'last_order_date': datetime.datetime(2025, 3, 5, 0, 0), 'average_amount': 87.5}, 102: {'total_orders': 1, 'total_amount_sum': 100, 'last_order_date': datetime.datetime(2025, 3, 2, 0, 0), 'average_amount': 100.0}}

All orders in broker 'orders' topic:
[{'order_id': 1, 'customer_id': 101, 'quantity': 2, 'unit_price': 50, 'order_date': '2025-03-01'}, {'order_id': 2, 'customer_id': 102, 'quantity': 1, 'unit_price': 100, 'order_date': '2025-03-02'}, {'order_id': 3, 'customer_id': 101, 'quantity': 1, 'unit_price': 75, 'order_date': '2025-03-05'}]


**Step 4:** Run 20 new orders through all three dataflow modes. Verify that the prediction service gets the same features in each case. Write a markdown cell comparing the three modes: coupling, latency characteristics, and what happens when one component goes down.

In [10]:
import random
from datetime import datetime, timedelta


random.seed(42)
new_orders = []
for i in range(1, 21):
    new_orders.append({
        "order_id": i,
        "customer_id": random.randint(100, 105),
        "quantity": random.randint(1, 5),
        "unit_price": round(random.uniform(10, 200), 2),
        "order_date": (datetime(2025, 3, 1) + timedelta(days=random.randint(0, 10))).strftime("%Y-%m-%d")
    })


database = {"orders": [], "features": {}}

def db_order_service(order):
    database["orders"].append(order)

def db_feature_service_compute():
    database["features"] = {}
    for order in database["orders"]:
        cid = order["customer_id"]
        total = order["quantity"] * order["unit_price"]
        date = datetime.strptime(order["order_date"], "%Y-%m-%d")
        f = database["features"].get(cid, {"total_orders":0, "total_amount_sum":0, "last_order_date":datetime.min})
        f["total_orders"] += 1
        f["total_amount_sum"] += total
        f["average_amount"] = f["total_amount_sum"] / f["total_orders"]
        f["last_order_date"] = max(f["last_order_date"], date)
        database["features"][cid] = f


for o in new_orders:
    db_order_service(o)
db_feature_service_compute()
db_features = database["features"]


features_storage = {}
def feature_service_get_features(order):
    cid = order["customer_id"]
    total = order["quantity"] * order["unit_price"]
    date = datetime.strptime(order["order_date"], "%Y-%m-%d")
    f = features_storage.get(cid, {"total_orders":0, "total_amount_sum":0, "last_order_date":datetime.min})
    f["total_orders"] += 1
    f["total_amount_sum"] += total
    f["average_amount"] = f["total_amount_sum"] / f["total_orders"]
    f["last_order_date"] = max(f["last_order_date"], date)
    features_storage[cid] = f
    return f

for o in new_orders:
    feature_service_get_features(o)
service_features = features_storage

class Broker:
    def __init__(self):
        self.topics = {}
    def publish(self, topic, message):
        self.topics.setdefault(topic, [])
        self.topics[topic].append(message)
        # notify subscribers
        for callback in getattr(self, f"{topic}_subscribers", []):
            callback(message)
    def subscribe(self, topic, callback):
        setattr(self, f"{topic}_subscribers", getattr(self, f"{topic}_subscribers", []))
        getattr(self, f"{topic}_subscribers").append(callback)

broker = Broker()
latest_features = {}

def feature_service_on_order(order):
    cid = order["customer_id"]
    total = order["quantity"] * order["unit_price"]
    date = datetime.strptime(order["order_date"], "%Y-%m-%d")
    f = latest_features.get(cid, {"total_orders":0, "total_amount_sum":0, "last_order_date":datetime.min})
    f["total_orders"] += 1
    f["total_amount_sum"] += total
    f["average_amount"] = f["total_amount_sum"] / f["total_orders"]
    f["last_order_date"] = max(f["last_order_date"], date)
    latest_features[cid] = f
    broker.publish("features", {cid: f})

def order_service_publish(order):
    broker.publish("orders", order)

broker.subscribe("orders", feature_service_on_order)


broker.subscribe("features", lambda f: None)  


for o in new_orders:
    order_service_publish(o)
broker_features = latest_features


def normalize_features(features):
    normalized = {}
    for cid, f in features.items():
        normalized[cid] = {
            "total_orders": f["total_orders"],
            "average_amount": round(f["average_amount"], 2),
            "last_order_date": f["last_order_date"].strftime("%Y-%m-%d") if isinstance(f["last_order_date"], datetime) else str(f["last_order_date"])
        }
    return normalized

db_norm = normalize_features(db_features)
service_norm = normalize_features(service_features)
broker_norm = normalize_features(broker_features)

assert db_norm == service_norm == broker_norm, "Feature mismatch across modes!"
print("✅ Features match across all three modes after normalization.")

example_cid = list(db_norm.keys())[0]
print("\nExample features for customer", example_cid)
print(db_norm[example_cid])

✅ Features match across all three modes after normalization.

Example features for customer 105
{'total_orders': 7, 'average_amount': 292.76, 'last_order_date': '2025-03-08'}


### Task 4: Batch Processing vs Stream Processing

**Step 1 — Batch processing:** Using the cleaned data from Task 1, write a batch processing function that computes daily aggregate features:
- Total orders per day
- Total revenue per day
- Average order size per day
- Number of unique customers per day
- Top product per day (by revenue)

Execute it and display the results.

In [11]:
import pandas as pd
def batch_aggregate(df):
    """
    Computes daily aggregates:
    - total orders per day
    - total revenue per day
    - average order size per day
    - unique customers per day
    - top product per day by revenue
    """
    df['order_date'] = pd.to_datetime(df['order_date'])
    

    daily = df.groupby(df['order_date'].dt.date)
    

    summary = daily.agg(
        total_orders=('order_id', 'count'),
        total_revenue=('total_amount', 'sum'),
        avg_order_size=('total_amount', 'mean'),
        unique_customers=('customer_id', pd.Series.nunique)
    ).reset_index()
    
    top_products = df.groupby([df['order_date'].dt.date, 'product_name']) \
                     .agg(daily_revenue=('total_amount', 'sum')) \
                     .reset_index()
    
    top_product_per_day = top_products.loc[top_products.groupby('order_date')['daily_revenue'].idxmax()]
    top_product_per_day = top_product_per_day[['order_date', 'product_name', 'daily_revenue']]
    
    summary = summary.merge(top_product_per_day, left_on='order_date', right_on='order_date')
    
    summary = summary.rename(columns={
        'product_name': 'top_product',
        'daily_revenue': 'top_product_revenue'
    })
    
    return summary

daily_summary = batch_aggregate(cleaned_df)
print("\nDaily Batch Summary:")
print(daily_summary)


Daily Batch Summary:
    order_date  total_orders  total_revenue  avg_order_size  unique_customers  \
0   2025-01-01             3        1438.92      479.640000                 3   
1   2025-01-02             2        1659.05      829.525000                 2   
2   2025-01-03             1         465.64      465.640000                 1   
3   2025-01-04             3        1194.53      398.176667                 3   
4   2025-01-05             4        4297.53     1074.382500                 4   
..         ...           ...            ...             ...               ...   
89  2025-04-25             3        1065.93      355.310000                 2   
90  2025-04-26             2        1469.74      734.870000                 2   
91  2025-04-27             1         297.75      297.750000                 1   
92  2025-04-29             5        4081.07      816.214000                 4   
93  2025-04-30             3         591.32      197.106667                 3   

   to

**Step 2 — Stream processing:** Implement a `StreamProcessor` class that processes orders one at a time and maintains running statistics:
- Running total of orders and revenue
- Windowed average (last 50 orders) of order size
- Running count of unique customers
- Current most popular product (by count in the last 50 orders)

Process the same data from Task 1 through the stream processor, one record at a time. After every 50 records, print the current streaming statistics.

In [12]:
from collections import deque, Counter

class StreamProcessor:
    def __init__(self, window_size=50):
        self.total_orders = 0
        self.total_revenue = 0.0
        self.unique_customers = set()
        self.window_size = window_size
        self.recent_orders = deque(maxlen=window_size)
        self.product_counter = Counter()
    
    def process_order(self, order):
        """Process a single order and update all statistics."""
        self.total_orders += 1
        self.total_revenue += order['total_amount']
        self.unique_customers.add(order['customer_id'])
        
        if len(self.recent_orders) == self.window_size:
            oldest_order = self.recent_orders.popleft()
            self.product_counter[oldest_order['product_name']] -= 1
            if self.product_counter[oldest_order['product_name']] == 0:
                del self.product_counter[oldest_order['product_name']]
        
        self.recent_orders.append(order)
        self.product_counter[order['product_name']] += 1
    
    def get_stats(self):
        """Return current streaming statistics."""
        if self.recent_orders:
            window_avg_order_size = sum(o['total_amount'] for o in self.recent_orders) / len(self.recent_orders)
        else:
            window_avg_order_size = 0.0
        
        if self.product_counter:
            most_popular_product = self.product_counter.most_common(1)[0][0]
        else:
            most_popular_product = None
        
        return {
            'total_orders': self.total_orders,
            'total_revenue': self.total_revenue,
            'unique_customers_count': len(self.unique_customers),
            'window_avg_order_size': window_avg_order_size,
            'most_popular_product_last_50': most_popular_product
        }

processor = StreamProcessor(window_size=50)

for i, order in enumerate(cleaned_df.to_dict(orient='records'), 1):
    processor.process_order(order)
    if i % 50 == 0:
        print(f"\nAfter {i} orders:")
        stats = processor.get_stats()
        for k, v in stats.items():
            print(f"{k}: {v:.2f}" if isinstance(v, float) else f"{k}: {v}")


After 50 orders:
total_orders: 50
total_revenue: 39640.19
unique_customers_count: 31
window_avg_order_size: 792.80
most_popular_product_last_50: Monitor

After 100 orders:
total_orders: 100
total_revenue: 70290.37
unique_customers_count: 42
window_avg_order_size: 613.00
most_popular_product_last_50: Mouse

After 150 orders:
total_orders: 150
total_revenue: 107464.89
unique_customers_count: 48
window_avg_order_size: 743.49
most_popular_product_last_50: Monitor


### Comparing Batch vs Stream Statistics

- The **overall totals** (total orders, total revenue, unique customers) from both batch and stream processing match, which means both methods are working correctly.  

- However, **windowed stream stats** (like average order size or most popular product in the last 50 orders) can differ from daily batch statistics because they only look at a recent subset of orders, not the full dataset.  

- Each approach gives you different insights:  
  - **Batch processing** – shows the full picture for a day or period: daily revenue, total orders, top products. Good for reporting and analysis.  
  - **Stream processing** – gives near real-time insights and highlights trends from recent orders. Useful for detecting spikes or reacting quickly.  

- **Takeaway:** Batch is stable and great for historical analysis, while streaming is excellent for real-time monitoring and fast decision-making.

### Task 5: Combine Batch and Stream Features

**Step 1:** Define a set of batch features and streaming features for a customer in an e-commerce system:

Batch features (computed from all historical data):
- `total_lifetime_orders`
- `avg_order_amount`
- `days_since_first_order`
- `most_purchased_category`




Streaming features (computed from last 10 orders):
- `recent_order_count`
- `recent_avg_amount`
- `seconds_since_last_order`
- `recent_top_category`

In [32]:
import pandas as pd
from datetime import datetime

cleaned_df['order_date'] = pd.to_datetime(cleaned_df['order_date'])

def compute_batch_features(df, customer_id):
    cust_orders = df[df['customer_id'] == customer_id]

    if cust_orders.empty:
        return {
            'customer_id': customer_id,
            'total_lifetime_orders': 0,
            'avg_order_amount': 0,
            'days_since_first_order': None,
            'most_purchased_category': None
        }
        
    total_lifetime_orders = len(cust_orders)
    avg_order_amount = cust_orders['total_amount'].mean()
    days_since_first_order = (datetime.now() - cust_orders['order_date'].min()).days
    most_purchased_category = cust_orders['product_name'].mode()[0] if not cust_orders['product_name'].mode().empty else None
    
    return {
        'customer_id': customer_id,
        'total_lifetime_orders': total_lifetime_orders,
        'avg_order_amount': avg_order_amount,
        'days_since_first_order': days_since_first_order,
        'most_purchased_category': most_purchased_category
    }


def compute_stream_features(df, customer_id, window_size=10):
    cust_orders = df[df['customer_id'] == customer_id].sort_values('order_date')

    if cust_orders.empty:
        return {
            'customer_id': customer_id,
            'recent_order_count': 0,
            'recent_avg_amount': 0,
            'seconds_since_last_order': None,
            'recent_top_category': None
        }

    recent_orders = cust_orders.tail(window_size)

    return {
        'customer_id': customer_id,
        'recent_order_count': len(recent_orders),
        'recent_avg_amount': recent_orders['total_amount'].mean(),
        'seconds_since_last_order': (datetime.now() - recent_orders['order_date'].max()).total_seconds(),
        'recent_top_category': recent_orders['product_name'].mode()[0] 
                               if not recent_orders['product_name'].mode().empty else None
    }
    

**Step 2:** For 5 sample customers, compute both batch and streaming features from the Task 1 data.

In [35]:
sample_customers = cleaned_df['customer_id'].unique()[:5]
all_features = []
for cid in sample_customers:
    batch = compute_batch_features(cleaned_df, cid)
    stream = compute_stream_features(cleaned_df, cid)
    all_features.append({"customer_id": cid, "batch": batch, "stream": stream})

for f in all_features:
    print(f"\nCustomer {f['customer_id']}:")
    print("  Batch features:", f["batch"])
    print("  Streaming features:", f["stream"])


Customer 4:
  Batch features: {'customer_id': np.int64(4), 'total_lifetime_orders': 6, 'avg_order_amount': np.float64(571.5566666666667), 'days_since_first_order': 429, 'most_purchased_category': 'Mouse'}
  Streaming features: {'customer_id': np.int64(4), 'recent_order_count': 6, 'recent_avg_amount': np.float64(571.5566666666667), 'seconds_since_last_order': 29605057.089249, 'recent_top_category': 'Mouse'}

Customer 3:
  Batch features: {'customer_id': np.int64(3), 'total_lifetime_orders': 7, 'avg_order_amount': np.float64(665.8114285714286), 'days_since_first_order': 410, 'most_purchased_category': 'Tablet'}
  Streaming features: {'customer_id': np.int64(3), 'recent_order_count': 7, 'recent_avg_amount': np.float64(665.8114285714286), 'seconds_since_last_order': 27099457.092178, 'recent_top_category': 'Tablet'}

Customer 11:
  Batch features: {'customer_id': np.int64(11), 'total_lifetime_orders': 7, 'avg_order_amount': np.float64(899.2385714285714), 'days_since_first_order': 411, 'mos

**Step 3:** Combine them into a single feature dictionary per customer. Print the combined feature vectors for all 5 customers.


In [37]:
combined_features_list = []

for cid in sample_customers:
    batch_feats = compute_batch_features(cleaned_df, cid)
    stream_feats = compute_stream_features(cleaned_df, cid)
    
    combined_feats = {**batch_feats, **stream_feats}
    combined_features_list.append(combined_feats)
    
    print(f"Customer {cid} combined features:")
    print(combined_feats)

Customer 4 combined features:
{'customer_id': np.int64(4), 'total_lifetime_orders': 6, 'avg_order_amount': np.float64(571.5566666666667), 'days_since_first_order': 429, 'most_purchased_category': 'Mouse', 'recent_order_count': 6, 'recent_avg_amount': np.float64(571.5566666666667), 'seconds_since_last_order': 29605238.532704, 'recent_top_category': 'Mouse'}
Customer 3 combined features:
{'customer_id': np.int64(3), 'total_lifetime_orders': 7, 'avg_order_amount': np.float64(665.8114285714286), 'days_since_first_order': 410, 'most_purchased_category': 'Tablet', 'recent_order_count': 7, 'recent_avg_amount': np.float64(665.8114285714286), 'seconds_since_last_order': 27099638.536757, 'recent_top_category': 'Tablet'}
Customer 11 combined features:
{'customer_id': np.int64(11), 'total_lifetime_orders': 7, 'avg_order_amount': np.float64(899.2385714285714), 'days_since_first_order': 411, 'most_purchased_category': 'Mouse', 'recent_order_count': 7, 'recent_avg_amount': np.float64(899.238571428571

### Why Combine Batch and Stream Features for ML Models

Using both **batch** and **streaming** features gives a machine learning model a fuller picture of customer behavior.

- **Batch features** summarize the customer’s entire history, like total lifetime orders or average order amount. They are great for understanding long-term habits, loyalty, or overall spending patterns.  
- **Stream features** capture recent behavior, like the last 10 orders or how recently a customer shopped. They are useful for detecting short-term changes, trends, or sudden spikes in activity.

**Example where batch features alone are insufficient:**  
Predicting whether a customer will make a purchase in the next day. A long-term average won’t capture a sudden recent interest spike or a recent browsing spree.

**Example where stream features alone are insufficient:**  
Predicting a customer’s lifetime value. Looking only at recent orders ignores the total historical spending and long-term trends, which are crucial for accurate estimation.

**Conclusion:** Combining both gives the model **stability from historical data** and **responsiveness to recent behavior**, improving prediction accuracy for both short-term and long-term tasks.